# Notebook 03 – Data Preprocessing

## Objective

This notebook prepares both datasets for machine learning.

### Tasks

- Clean the Jigsaw toxicity dataset
- Clean the Social Media dataset
- Normalize text
- Remove missing values
- Handle duplicate posts
- Create binary toxicity labels
- Detect gender/race keywords
- Save processed datasets

Output:

- Dataset/processed/jigsaw_processed.parquet
- Dataset/processed/social_processed.parquet

In [1]:
from pathlib import Path

import re
import emoji
import ftfy
import nltk
import numpy as np
import pandas as pd

from cleantext import clean
from tqdm.auto import tqdm

tqdm.pandas()

nltk.download("stopwords")
nltk.download("punkt")

Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.
d:\Projects\sentiment-analysis-socialmedia\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tzmughal\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tzmughal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
DATASET = Path("../Dataset")

TRAIN_PATH = DATASET / "train.csv"
SOCIAL_PATH = DATASET / "merged"

OUTPUT = DATASET / "processed"
OUTPUT.mkdir(exist_ok=True)

print(OUTPUT)

..\Dataset\processed


In [3]:
jigsaw = pd.read_csv(TRAIN_PATH)

print(jigsaw.shape)
jigsaw.head()

(1804874, 45)


,id,target,comment_text,severe_toxicity,obscene,identity_attack,insult,threat,asian,atheist,...,article_id,rating,funny,wow,sad,likes,disagree,sexual_explicit,identity_annotator_count,toxicity_annotator_count
0,59848,0.000000,"This is so cool. It's like, 'would you want yo...",0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
1,59849,0.000000,Thank you!! This would make my life a lot less...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
2,59852,0.000000,This is such an urgent design problem; kudos t...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
3,59855,0.000000,Is this something I'll be able to install on m...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
4,59856,0.893617,haha you guys are a bunch of losers.,0.021277,0.0,0.021277,0.87234,0.0,0.0,0.0,...,2006,rejected,0,0,0,1,0,0.0,4,47


In [4]:
files = sorted(SOCIAL_PATH.glob("*.jsonl"))

len(files)

11

In [5]:
social = pd.concat(
    (
        pd.read_json(
            f,
            lines=True,
            dtype={
                "text":"string",
                "instance":"string",
                "user_id":"Int64"
            }
        )[[
            "post_id",
            "user_id",
            "instance",
            "date",
            "text",
            "sent_label",
            "sent_score"
        ]]
        for f in tqdm(files)
    ),
    ignore_index=True
)

social.shape

100%|██████████| 11/11 [06:45<00:00, 36.86s/it]


(19508046, 7)

In [6]:
print(jigsaw.shape)

jigsaw = jigsaw.dropna(subset=["comment_text"])

print(jigsaw.shape)

(1804874, 45)
(1804871, 45)


In [7]:
print(social.shape)

social = social.dropna(subset=["text"])

print(social.shape)

(19508046, 7)
(19508046, 7)


## Remove Duplicate Text

In [8]:
print(jigsaw.shape)

jigsaw = jigsaw.drop_duplicates(subset="comment_text")

print(jigsaw.shape)

(1804871, 45)
(1780822, 45)


In [9]:
print(social.shape)

social = social.drop_duplicates(subset="text")

print(social.shape)

(19508046, 7)
(15223017, 7)


## Text Cleaning

In [10]:
URL = re.compile(r"http\\S+|www\\.\\S+")

In [11]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = ftfy.fix_text(str(text))

    text = emoji.demojize(text)

    text = URL.sub(" ", text)

    text = clean(
        text,
        lower=True,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_currency_symbols=True,
        replace_with_url="",
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [12]:
jigsaw["clean_text"] = jigsaw["comment_text"].progress_apply(clean_text)

100%|██████████| 1780822/1780822 [28:49<00:00, 1029.95it/s]


In [13]:
social["clean_text"] = social["text"].progress_apply(clean_text)

100%|██████████| 15223017/15223017 [1:53:31<00:00, 2234.80it/s]  


## Toxicity Labels

In [14]:
jigsaw["is_toxic"] = (jigsaw["target"] >= 0.5).astype(int)

jigsaw["is_toxic"].value_counts()

is_toxic
0    1638373
1     142449
Name: count, dtype: int64

## Dataset Summary

In [17]:
print(jigsaw.shape)
print(social.shape)

(1780822, 47)
(15223017, 8)


## Save Processed Data

In [18]:
jigsaw.to_parquet(
    OUTPUT / "jigsaw_processed.parquet",
    index=False
)

social.to_parquet(
    OUTPUT / "social_processed.parquet",
    index=False
)

In [19]:
print("Saved!")

print(OUTPUT / "jigsaw_processed.parquet")
print(OUTPUT / "social_processed.parquet")

Saved!
..\Dataset\processed\jigsaw_processed.parquet
..\Dataset\processed\social_processed.parquet


In [20]:
from pathlib import Path
import pandas as pd

OUTPUT = Path("../Dataset/processed")

jigsaw = pd.read_parquet(OUTPUT / "jigsaw_processed.parquet")
social = pd.read_parquet(OUTPUT / "social_processed.parquet")

print("Jigsaw:", jigsaw.shape)
print("Social:", social.shape)

print(jigsaw.columns.tolist())
print(social.columns.tolist())

Jigsaw: (1780822, 47)
Social: (15223017, 8)
['id', 'target', 'comment_text', 'severe_toxicity', 'obscene', 'identity_attack', 'insult', 'threat', 'asian', 'atheist', 'bisexual', 'black', 'buddhist', 'christian', 'female', 'heterosexual', 'hindu', 'homosexual_gay_or_lesbian', 'intellectual_or_learning_disability', 'jewish', 'latino', 'male', 'muslim', 'other_disability', 'other_gender', 'other_race_or_ethnicity', 'other_religion', 'other_sexual_orientation', 'physical_disability', 'psychiatric_or_mental_illness', 'transgender', 'white', 'created_date', 'publication_id', 'parent_id', 'article_id', 'rating', 'funny', 'wow', 'sad', 'likes', 'disagree', 'sexual_explicit', 'identity_annotator_count', 'toxicity_annotator_count', 'clean_text', 'is_toxic']
['post_id', 'user_id', 'instance', 'date', 'text', 'sent_label', 'sent_score', 'clean_text']


## Summary

### Jigsaw Dataset

- Missing comments removed
- Duplicate comments removed
- Text normalized
- Binary toxicity label created
- Saved as Parquet

### Social Media Dataset

- Missing posts removed
- Duplicate posts removed
- Text normalized
- Saved as Parquet

### Notes

Keyword-based gender/race analysis will be performed after model inference to ensure the analysis reflects predicted toxicity scores rather than influencing model training.

# Prepare Balanced Training Dataset

The processed Jigsaw dataset still contains many metadata columns that are unnecessary for transformer training.

This section prepares a lightweight dataset specifically for model training by:

- Loading the processed dataset
- Keeping only the required columns
- Removing invalid samples
- Balancing the classes
- Creating Train / Validation / Test splits
- Saving the final datasets for Notebook 04

This preprocessing is performed only once and significantly reduces memory usage during model training.

In [21]:
from pathlib import Path
from sklearn.model_selection import train_test_split
import json

## Load Processed Dataset

Load the processed Jigsaw dataset generated earlier in this notebook.

In [2]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()

PROJECT_DIR = NOTEBOOK_DIR.parent

DATASET_DIR = PROJECT_DIR / "Dataset"

PROCESSED_DIR = DATASET_DIR / "processed"

print(PROCESSED_DIR)

d:\Projects\sentiment-analysis-socialmedia\Dataset\processed


In [3]:
jigsaw_path = PROCESSED_DIR / "jigsaw_processed.parquet"

social_path = PROCESSED_DIR / "social_processed.parquet"

print(jigsaw_path.exists())
print(social_path.exists())

True
True


# Create Training Dataset



For efficient model training, we create a lightweight dataset containing only the required columns.

Required columns:

- clean_text
- is_toxic

This dataset will be used directly in Notebook 04.

In [4]:
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent

processed_dir = PROJECT_DIR / "Dataset" / "processed"

processed_path = processed_dir / "jigsaw_processed.parquet"

training_df = pd.read_parquet(
    processed_path,
    columns=["clean_text", "is_toxic"]
)

print(training_df.shape)

training_df.head()

(1780822, 2)


,clean_text,is_toxic
0,"this is so cool. it's like, 'would you want yo...",0
1,thank you!! this would make my life a lot less...,0
2,this is such an urgent design problem; kudos t...,0
3,is this something i'll be able to install on m...,0
4,haha you guys are a bunch of losers.,1


# Create Balanced Training Dataset 

Instead of loading the entire dataset into memory, we process the parquet file in batches using PyArrow.

Advantages:

- Low RAM usage
- Faster processing
- No ArrowMemoryError
- Reproducible sampling

In [5]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

In [6]:
NOTEBOOK_DIR = Path.cwd()

PROJECT_DIR = NOTEBOOK_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "Dataset" / "processed"

PARQUET_FILE = PROCESSED_DIR / "jigsaw_processed.parquet"

print(PARQUET_FILE)

d:\Projects\sentiment-analysis-socialmedia\Dataset\processed\jigsaw_processed.parquet


# Read Only Required Columns

Only the columns required for model training are loaded:

- clean_text
- is_toxic

This reduces memory usage significantly.

In [7]:
parquet = pq.ParquetFile(PARQUET_FILE)

print("Row Groups :", parquet.num_row_groups)

Row Groups : 2


# Collect Positive and Negative Samples

Each row group is processed independently.

Only the required rows are kept before moving to the next batch.

In [8]:
positive_parts = []

negative_parts = []

for rg in range(parquet.num_row_groups):

    batch = parquet.read_row_group(
        rg,
        columns=[
            "clean_text",
            "is_toxic"
        ]
    )

    df = batch.to_pandas()

    positive_parts.append(
        df[df.is_toxic == 1]
    )

    negative_parts.append(
        df[df.is_toxic == 0]
    )

    print(
        f"Processed row group {rg+1}/{parquet.num_row_groups}"
    )

Processed row group 1/2
Processed row group 2/2


In [9]:
positive_df = pd.concat(
    positive_parts,
    ignore_index=True
)

negative_df = pd.concat(
    negative_parts,
    ignore_index=True
)

print(len(positive_df))
print(len(negative_df))

142449
1638373


# Random Sampling

To reduce CPU training time, we randomly select an equal number of toxic and non-toxic comments.

Default sample size:

- 50,000 toxic
- 50,000 non-toxic

Total:

100,000 comments

In [10]:
SEED = 42

SAMPLE_SIZE = 50000

positive_sample = positive_df.sample(
    n=min(SAMPLE_SIZE, len(positive_df)),
    random_state=SEED
)

negative_sample = negative_df.sample(
    n=min(SAMPLE_SIZE, len(negative_df)),
    random_state=SEED
)

In [11]:
balanced_df = pd.concat(
    [
        positive_sample,
        negative_sample
    ],
    ignore_index=True
)

balanced_df = balanced_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

balanced_df.head()

,clean_text,is_toxic
0,nice one eve,0
1,in my opinion trump is in the presidency to pr...,0
2,do you really need research to know this is a ...,1
3,"teaching about american history, free-maket ec...",0
4,"well, it happens when you put race before perf...",0


In [12]:
balanced_df.is_toxic.value_counts()

is_toxic
0    50000
1    50000
Name: count, dtype: int64

# Save Balanced Dataset

The balanced dataset is stored for direct use in Notebook 04.

No further preprocessing will be required during model training.

In [13]:
balanced_path = PROCESSED_DIR / "jigsaw_balanced.parquet"

balanced_df.to_parquet(
    balanced_path,
    index=False
)

print(balanced_path)

d:\Projects\sentiment-analysis-socialmedia\Dataset\processed\jigsaw_balanced.parquet


# Create Train / Validation Split

The balanced dataset is split into training and validation sets using a stratified split.

Saving these files ensures:

- Reproducibility
- Faster training startup
- Consistent evaluation across experiments

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
train_df, valid_df = train_test_split(
    balanced_df,
    test_size=0.20,
    random_state=SEED,
    stratify=balanced_df["is_toxic"],
)

print(train_df.shape)
print(valid_df.shape)

(80000, 2)
(20000, 2)


In [16]:
split_dir = PROCESSED_DIR / "train_valid_split"

split_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [17]:
train_path = split_dir / "train.parquet"
valid_path = split_dir / "valid.parquet"

train_df.to_parquet(
    train_path,
    index=False
)

valid_df.to_parquet(
    valid_path,
    index=False
)

print(train_path)
print(valid_path)

d:\Projects\sentiment-analysis-socialmedia\Dataset\processed\train_valid_split\train.parquet
d:\Projects\sentiment-analysis-socialmedia\Dataset\processed\train_valid_split\valid.parquet


In [18]:
print(train_df["is_toxic"].value_counts())

print()

print(valid_df["is_toxic"].value_counts())

is_toxic
0    40000
1    40000
Name: count, dtype: int64

is_toxic
1    10000
0    10000
Name: count, dtype: int64
